# ML Meta Cluster

Ноутбук проверяет cluster meta-признаки для baseline ML-моделей.
Для каждой модели сравниваются несколько способов кластеризации, а holdout используется только для финальной оценки.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering, Birch, DBSCAN, OPTICS, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import pairwise_distances

from xgboost import XGBRegressor

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

SEED = 2
TARGET_COL = "duration_hours"
DURATION_CAP = 960
HOLDOUT_FRAC = 0.20
CV_SPLITS = 3
LOGREG_OOF_SPLITS = 5
USE_PCA = True
PCA_COMPONENTS = 30

DATA_PATH_ENV = "DATA_FINALL_PATH"
ARTIFACT_DIR_ENV = "META_ARTIFACTS_DIR"


def require_path(env_name: str, label: str) -> Path:
    value = os.environ.get(env_name)
    if not value:
        raise RuntimeError(f"Укажи путь к {label} в переменной окружения {env_name}.")
    path = Path(value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
ARTIFACT_DIR = Path(os.environ.get(ARTIFACT_DIR_ENV, "./artifacts_meta_all_baseline_bigsearch_optuna")).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def normalized_entropy(proba):
    proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.zeros(len(proba), dtype=float)
    ent = -(proba * np.log(proba)).sum(axis=1)
    return ent / np.log(proba.shape[1])

def top2_margin(proba):
    proba = np.asarray(proba, dtype=float)
    if proba.ndim != 2 or proba.shape[1] <= 1:
        return np.ones(len(proba), dtype=float)
    desc = np.sort(proba, axis=1)[:, ::-1]
    return desc[:, 0] - desc[:, 1]

def json_ready(obj):
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return [json_ready(v) for v in obj.tolist()]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    return obj

def load_split_data():
    df = pd.read_csv(DATA_PATH)
    split = int(len(df) * (1 - HOLDOUT_FRAC))

    train_full = df.iloc[:split].copy().reset_index(drop=True)
    holdout_full = df.iloc[split:].copy().reset_index(drop=True)

    train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
    holdout_typical = holdout_full[holdout_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

    return df, train_full, train_filt, holdout_full, holdout_typical

BASELINE_PIPELINES = [
    ("Scaled_Lasso", Pipeline([
        ("Scaler", StandardScaler()),
        ("Lasso", Lasso(random_state=SEED))
    ])),
    ("Scaled_Elastic", Pipeline([
        ("Scaler", StandardScaler()),
        ("Elastic", ElasticNet(random_state=SEED))
    ])),
    ("Scaled_RF_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_ET_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("ET", ExtraTreesRegressor(random_state=SEED))
    ])),
    ("Scaled_BR_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BaggingRegressor(random_state=SEED))
    ])),
    ("Scaled_DT_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("DT_reg", DecisionTreeRegressor(random_state=SEED))
    ])),
    ("Scaled_Ridge", Pipeline([
        ("Scaler", StandardScaler()),
        ("Ridge", Ridge(random_state=SEED))
    ])),
    ("Scaled_SVR", Pipeline([
        ("Scaler", StandardScaler()),
        ("SVR", SVR(kernel="linear", C=1e2))
    ])),
    ("Scaled_Hub-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("Hub-Reg", HuberRegressor())
    ])),
    ("Scaled_BayRidge", Pipeline([
        ("Scaler", StandardScaler()),
        ("BR", BayesianRidge())
    ])),
    ("Scaled_KNN_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("KNN_reg", KNeighborsRegressor())
    ])),
    ("Scaled_Gboost-Reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("GBoost-Reg", GradientBoostingRegressor(random_state=SEED))
    ])),
    ("Scaled_XGB_reg", Pipeline([
        ("Scaler", StandardScaler()),
        ("XGBR", XGBRegressor(random_state=SEED))
    ])),
    ("Scaled_RFR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("RF", RandomForestRegressor(random_state=SEED))
    ])),
    ("Scaled_XGBR_PCA", Pipeline([
        ("Scaler", StandardScaler()),
        ("PCA", PCA(n_components=3)),
        ("XGB", XGBRegressor(random_state=SEED))
    ])),
]

MODEL_ORDER = [name for name, _ in BASELINE_PIPELINES]
BASELINE_PIPELINE_LOOKUP = {name: model for name, model in BASELINE_PIPELINES}

def build_regression_model(model_name, extra_params=None):
    if model_name not in BASELINE_PIPELINE_LOOKUP:
        raise ValueError(model_name)

    est = clone(BASELINE_PIPELINE_LOOKUP[model_name])

    if extra_params is not None:
        current = est.get_params()
        safe_updates = {k: v for k, v in extra_params.items() if k in current}
        if safe_updates:
            est.set_params(**safe_updates)

    return est

def compute_baseline_cv_and_holdout(model_names, train_filt, holdout_typical, holdout_full):
    rows = []
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

    Xtrain = train_filt.drop(columns=[TARGET_COL]).reset_index(drop=True)
    ytrain = train_filt[TARGET_COL].reset_index(drop=True)

    Xhold_typ = holdout_typical.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_typ = holdout_typical[TARGET_COL].reset_index(drop=True)

    Xhold_full = holdout_full.drop(columns=[TARGET_COL]).reset_index(drop=True)
    yhold_full = holdout_full[TARGET_COL].reset_index(drop=True)

    for model_name in model_names:
        fold_maes = []

        for tr_idx, va_idx in tscv.split(train_filt):
            fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
            fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

            Xtr = fold_train.drop(columns=[TARGET_COL]).reset_index(drop=True)
            ytr = fold_train[TARGET_COL].reset_index(drop=True)

            Xva = fold_valid.drop(columns=[TARGET_COL]).reset_index(drop=True)
            yva = fold_valid[TARGET_COL].reset_index(drop=True)

            est = build_regression_model(model_name)
            est.fit(Xtr, ytr)
            pred = est.predict(Xva)
            fold_maes.append(float(mean_absolute_error(yva, pred)))

        est_full = build_regression_model(model_name)
        est_full.fit(Xtrain, ytrain)

        pred_typ = est_full.predict(Xhold_typ)
        pred_full = est_full.predict(Xhold_full)

        m_typ = reg_metrics(yhold_typ, pred_typ)
        m_full = reg_metrics(yhold_full, pred_full)

        rows.append({
            "model": model_name,
            "baseline_cv_MAE": round(float(np.mean(fold_maes)), 4),
            "baseline_cv_std": round(float(np.std(fold_maes)), 4),
            "baseline_holdout_typical_MAE": round(m_typ["MAE"], 4),
            "baseline_holdout_typical_RMSE": round(m_typ["RMSE"], 4),
            "baseline_holdout_typical_MdAE": round(m_typ["MdAE"], 4),
            "baseline_holdout_full_MAE": round(m_full["MAE"], 4),
            "baseline_holdout_full_RMSE": round(m_full["RMSE"], 4),
            "baseline_holdout_full_MdAE": round(m_full["MdAE"], 4),
        })

    return pd.DataFrame(rows).sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)

all_df, train_full, train_filt, holdout_full, holdout_typical = load_split_data()

baseline_cache_path = ARTIFACT_DIR / "baseline_summary_all_baseline_bigsearch_optuna.csv"
if baseline_cache_path.exists():
    baseline_summary = pd.read_csv(baseline_cache_path)
    baseline_summary = baseline_summary.sort_values(["baseline_cv_MAE", "baseline_cv_std"]).reset_index(drop=True)
    print(f"Baseline summary rows: {len(baseline_summary)}")
else:
    baseline_summary = compute_baseline_cv_and_holdout(MODEL_ORDER, train_filt, holdout_typical, holdout_full)
    baseline_summary.to_csv(baseline_cache_path, index=False)
    print(f"Baseline summary rows: {len(baseline_summary)}")

print("rows total =", len(all_df))
print("train rows =", len(train_full))
print("train typical rows =", len(train_filt))
print("holdout rows =", len(holdout_full))
print("holdout typical rows =", len(holdout_typical))
print("models =", len(MODEL_ORDER))
display(baseline_summary)


In [ ]:

rng = np.random.default_rng(SEED)
CLUSTER_MIN_VALID_CLUSTERS = 2
CLUSTER_MAX_VALID_CLUSTERS = 60
CLUSTER_MAX_HEAVY_FIT_ROWS = 3000

CLUSTER_EXPERIMENTS = {
    "MiniBatchKMeans": [
        {"n_clusters": k}
        for k in [2, 3, 4, 5, 6, 8, 10, 12]
    ],
    "GaussianMixture": [
        {"n_components": k, "covariance_type": cov}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for cov in ["full", "diag", "tied"]
    ],
    "Ward": [
        {"n_clusters": k}
        for k in [2, 3, 4, 5, 6, 8, 10]
    ],
    "AgglomerativeClustering": [
        {"n_clusters": k, "linkage": linkage}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for linkage in ["complete", "average"]
    ],
    "DBSCAN": [
        {"eps": eps, "min_samples": ms}
        for eps in [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]
        for ms in [5, 10, 20]
    ],
    "OPTICS": [
        {"min_samples": ms, "xi": xi}
        for ms in [5, 10, 20]
        for xi in [0.03, 0.05, 0.10]
    ],
    "BIRCH": [
        {"n_clusters": k, "threshold": th}
        for k in [2, 3, 4, 5, 6, 8, 10]
        for th in [0.2, 0.3, 0.5, 0.7]
    ],
    "SpectralClustering": [
        {"n_clusters": k, "affinity": "nearest_neighbors"}
        for k in [2, 3, 4, 5, 6]
    ],
}

if HDBSCAN_AVAILABLE:
    CLUSTER_EXPERIMENTS["HDBSCAN"] = [
        {"min_cluster_size": mcs, "min_samples": ms}
        for mcs in [5, 10, 20, 30]
        for ms in [None, 5, 10]
    ]

VERBOSE_CLUSTER_RUNS = True
SAVE_CLUSTER_PROGRESS_EVERY = 20

def stable_softmax_negdist(dist):
    dist = np.asarray(dist, dtype=float)
    scale = np.std(dist) if np.std(dist) > 1e-12 else 1.0
    z = -dist / scale
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def probs_from_dist(all_d):
    return stable_softmax_negdist(all_d)

def fit_preprocessor(X_train):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)

    pca = None
    Xt = Xs
    if USE_PCA and Xs.shape[1] > PCA_COMPONENTS:
        pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
        Xt = pca.fit_transform(Xs)

    return scaler, pca, Xt

def apply_preprocessor(X, scaler, pca):
    Xs = scaler.transform(X)
    return pca.transform(Xs) if pca is not None else Xs

def build_centroids(X, labels):
    labels = np.asarray(labels, dtype=int)
    valid_mask = labels != -1
    valid_labels = np.unique(labels[valid_mask])

    if len(valid_labels) < CLUSTER_MIN_VALID_CLUSTERS:
        return None, None

    label_map = {old: new for new, old in enumerate(sorted(valid_labels))}
    mapped = np.full_like(labels, -1)

    centroids = []
    for old_label in sorted(valid_labels):
        m = labels == old_label
        mapped[m] = label_map[old_label]
        centroids.append(X[m].mean(axis=0))

    centroids = np.vstack(centroids)
    if len(centroids) > CLUSTER_MAX_VALID_CLUSTERS:
        return None, None

    return centroids, mapped

def assign_by_centroids(X, centroids):
    all_d = pairwise_distances(X, centroids, metric="euclidean")
    labels = np.argmin(all_d, axis=1)
    dmin = all_d[np.arange(len(X)), labels]
    return labels, dmin, all_d

def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=SEED,
            n_init=20,
            batch_size=1024
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=SEED
        )
    elif name == "Ward":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage="ward"
        )
    elif name == "AgglomerativeClustering":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage=params["linkage"]
        )
    elif name == "DBSCAN":
        return DBSCAN(
            eps=params["eps"],
            min_samples=params["min_samples"]
        )
    elif name in {"BIRCH", "Birch"}:
        return Birch(
            n_clusters=params["n_clusters"],
            threshold=params["threshold"]
        )
    elif name == "SpectralClustering":
        return SpectralClustering(
            n_clusters=params["n_clusters"],
            affinity=params["affinity"],
            random_state=SEED,
            assign_labels="kmeans"
        )
    elif name == "OPTICS":
        return OPTICS(
            min_samples=params["min_samples"],
            xi=params["xi"]
        )
    elif name == "HDBSCAN":
        return hdbscan.HDBSCAN(
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
            prediction_data=True
        )
    else:
        raise ValueError(name)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    Xfit = Xtr
    fit_mode = "full_train"

    if name in {"SpectralClustering", "HDBSCAN", "OPTICS", "DBSCAN"} and len(Xtr) > CLUSTER_MAX_HEAVY_FIT_ROWS:
        idx = rng.choice(len(Xtr), size=CLUSTER_MAX_HEAVY_FIT_ROWS, replace=False)
        idx = np.sort(idx)
        Xfit = Xtr[idx]
        fit_mode = f"subsample_{CLUSTER_MAX_HEAVY_FIT_ROWS}"

    est = make_clusterer(name, params)

    if name == "GaussianMixture":
        est.fit(Xfit)
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    if name == "HDBSCAN":
        est.fit(Xfit)
        fit_labels = est.labels_
        if len(np.unique(fit_labels[fit_labels != -1])) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        try:
            te_labels, _ = hdbscan.approximate_predict(est, Xte)
        except Exception:
            return None

        centroids, _ = build_centroids(Xfit, fit_labels)
        if centroids is None:
            return None

        tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
        te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native_test"

    if hasattr(est, "fit_predict"):
        fit_labels = est.fit_predict(Xfit)
    else:
        est.fit(Xfit)
        fit_labels = est.labels_ if hasattr(est, "labels_") else est.predict(Xfit)

    if hasattr(est, "predict") and name in {"MiniBatchKMeans", "BIRCH", "Birch"} and len(Xfit) == len(Xtr):
        tr_labels = est.predict(Xtr)
        te_labels = est.predict(Xte)

        if hasattr(est, "cluster_centers_"):
            tr_d = pairwise_distances(Xtr, est.cluster_centers_)
            te_d = pairwise_distances(Xte, est.cluster_centers_)
            tr_proba = probs_from_dist(tr_d)
            te_proba = probs_from_dist(te_d)
        else:
            tr_proba = te_proba = None

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    centroids, _ = build_centroids(Xfit, fit_labels)
    if centroids is None:
        return None

    tr_labels, tr_dmin, tr_d = assign_by_centroids(Xtr, centroids)
    te_labels, te_dmin, te_d = assign_by_centroids(Xte, centroids)
    tr_proba = probs_from_dist(tr_d)
    te_proba = probs_from_dist(te_d)

    if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
        return None

    return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_centroid"

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["cluster_is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["cluster_is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster")
    te_ohe = te_ohe.reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        tr_feat["cluster_max_proba"] = np.max(tr_proba, axis=1)
        te_feat["cluster_max_proba"] = np.max(te_proba, axis=1)

        tr_feat["cluster_entropy"] = normalized_entropy(tr_proba)
        te_feat["cluster_entropy"] = normalized_entropy(te_proba)

        tr_feat["cluster_margin_top1_top2"] = top2_margin(tr_proba)
        te_feat["cluster_margin_top1_top2"] = top2_margin(te_proba)

        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]
    else:
        tr_feat["cluster_max_proba"] = 1.0
        te_feat["cluster_max_proba"] = 1.0
        tr_feat["cluster_entropy"] = 0.0
        te_feat["cluster_entropy"] = 0.0
        tr_feat["cluster_margin_top1_top2"] = 1.0
        te_feat["cluster_margin_top1_top2"] = 1.0

    return tr_feat, te_feat

def build_cluster_augmented(train_df, infer_df, cluster_name, cluster_params):
    train_df = train_df.reset_index(drop=True)
    infer_df = infer_df.reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET_COL]).reset_index(drop=True)
    X_infer = infer_df.drop(columns=[TARGET_COL]).reset_index(drop=True)

    scaler, pca, Xt_train = fit_preprocessor(X_train)
    Xt_infer = apply_preprocessor(X_infer, scaler, pca)

    result = fit_clusterer_and_assign(cluster_name, cluster_params, Xt_train, Xt_infer)
    if result is None:
        return None

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    X_train_aug = pd.concat([X_train, tr_feat.reset_index(drop=True)], axis=1)
    X_infer_aug = pd.concat([X_infer, te_feat.reset_index(drop=True)], axis=1)

    meta_info = {
        "cluster_name": cluster_name,
        "cluster_params": json_ready(cluster_params),
        "n_clusters_train": int(len(pd.Series(tr_labels).unique())),
        "use_pca": bool(USE_PCA),
        "pca_components": int(PCA_COMPONENTS) if (USE_PCA and X_train.shape[1] > PCA_COMPONENTS) else None,
        "fit_mode": fit_mode,
        "n_meta_features": int(tr_feat.shape[1]),
    }
    return X_train_aug, X_infer_aug, meta_info


print("Cluster meta screening")

total_cluster_cfg = sum(len(v) for v in CLUSTER_EXPERIMENTS.values())
print("models =", len(MODEL_ORDER))
print("cluster methods =", len(CLUSTER_EXPERIMENTS))
print("cluster configs =", total_cluster_cfg)
print("runs =", len(MODEL_ORDER) * total_cluster_cfg)

tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
screen_rows = []
config_summary_rows = []
run_id = 0
total_runs = len(MODEL_ORDER) * total_cluster_cfg

for cluster_name, cfg_list in CLUSTER_EXPERIMENTS.items():
    print(f"\n{cluster_name}: {len(cfg_list)} configs")

    for cluster_params in cfg_list:
        t0 = time.time()
        per_model_rows = []
        per_model_deltas = []
        per_model_maes = []

        for model_name in MODEL_ORDER:
            base_row = baseline_summary[baseline_summary["model"] == model_name].iloc[0]
            fold_maes = []
            valid_run = True
            last_meta_info = None

            for tr_idx, va_idx in tscv.split(train_filt):
                fold_train = train_filt.iloc[tr_idx].copy().reset_index(drop=True)
                fold_valid = train_filt.iloc[va_idx].copy().reset_index(drop=True)

                try:
                    built = build_cluster_augmented(fold_train, fold_valid, cluster_name, cluster_params)
                except Exception:
                    built = None

                if built is None:
                    valid_run = False
                    break

                Xtr_aug, Xva_aug, meta_info = built
                ytr = fold_train[TARGET_COL].reset_index(drop=True)
                yva = fold_valid[TARGET_COL].reset_index(drop=True)

                try:
                    est = build_regression_model(model_name)
                    est.fit(Xtr_aug, ytr)
                    pred = est.predict(Xva_aug)
                    fold_maes.append(float(mean_absolute_error(yva, pred)))
                    last_meta_info = meta_info
                except Exception:
                    valid_run = False
                    break

            run_id += 1

            if not valid_run or len(fold_maes) != CV_SPLITS:
                if VERBOSE_CLUSTER_RUNS:
                    print(f"[skip {run_id}/{total_runs}] {model_name} | {cluster_name}")
                continue

            cv_mae = float(np.mean(fold_maes))
            cv_std = float(np.std(fold_maes))
            baseline_cv = float(base_row["baseline_cv_MAE"])
            delta_cv = baseline_cv - cv_mae

            row = {
                "model": model_name,
                "cluster_name": cluster_name,
                "cluster_params_json": json.dumps(json_ready(cluster_params), ensure_ascii=False, sort_keys=True),
                "cv_typical_MAE": round(cv_mae, 4),
                "cv_typical_std": round(cv_std, 4),
                "baseline_cv_MAE": baseline_cv,
                "delta_cv_typical": round(delta_cv, 4),
                "n_clusters_train": int(last_meta_info["n_clusters_train"]),
                "use_pca": bool(last_meta_info["use_pca"]),
                "pca_components": last_meta_info["pca_components"],
                "fit_mode": last_meta_info["fit_mode"],
                "n_meta_features": int(last_meta_info["n_meta_features"]),
            }
            screen_rows.append(row)
            per_model_rows.append(row)
            per_model_deltas.append(delta_cv)
            per_model_maes.append(cv_mae)

        if len(per_model_rows) > 0:
            avg_delta = float(np.mean(per_model_deltas))
            med_delta = float(np.median(per_model_deltas))
            best_delta = float(np.max(per_model_deltas))
            avg_cv = float(np.mean(per_model_maes))
            elapsed = time.time() - t0

            config_summary_rows.append({
                "cluster_name": cluster_name,
                "cluster_params_json": json.dumps(json_ready(cluster_params), ensure_ascii=False, sort_keys=True),
                "avg_delta_cv_typical": round(avg_delta, 4),
                "median_delta_cv_typical": round(med_delta, 4),
                "best_delta_cv_typical": round(best_delta, 4),
                "avg_cv_typical_MAE": round(avg_cv, 4),
                "valid_models": len(per_model_rows),
                "elapsed_sec": round(elapsed, 2),
            })

            if VERBOSE_CLUSTER_RUNS:
                print(
                    f"{cluster_name} {cluster_params} | avg Δcv={avg_delta:+.4f} | "
                    f"med Δcv={med_delta:+.4f} | best Δcv={best_delta:+.4f} | "
                    f"avg cv={avg_cv:.4f} | valid_models={len(per_model_rows)} | {elapsed:.1f}s"
                )

        if (len(config_summary_rows) % SAVE_CLUSTER_PROGRESS_EVERY == 0) and len(screen_rows) > 0:
            pd.DataFrame(screen_rows).to_csv(
                ARTIFACT_DIR / "04_cluster_screening_all_baseline_bigsearch_optuna_progress.csv",
                index=False
            )
            pd.DataFrame(config_summary_rows).sort_values(
                ["avg_delta_cv_typical", "best_delta_cv_typical"],
                ascending=[False, False]
            ).to_csv(
                ARTIFACT_DIR / "04_cluster_config_summary_all_baseline_bigsearch_optuna_progress.csv",
                index=False
            )

cluster_screen_df = pd.DataFrame(screen_rows).sort_values(
    ["cv_typical_MAE", "cv_typical_std", "delta_cv_typical"],
    ascending=[True, True, False],
).reset_index(drop=True)

cluster_config_summary_df = pd.DataFrame(config_summary_rows).sort_values(
    ["avg_delta_cv_typical", "best_delta_cv_typical"],
    ascending=[False, False],
).reset_index(drop=True)

display(cluster_config_summary_df.head(20))
display(cluster_screen_df.head(30))


In [ ]:

best_rows = []

Xtrain = train_filt.drop(columns=[TARGET_COL]).reset_index(drop=True)
ytrain = train_filt[TARGET_COL].reset_index(drop=True)

for model_name in MODEL_ORDER:
    sub = cluster_screen_df[cluster_screen_df["model"] == model_name].copy()
    if sub.empty:
        continue

    best = sub.iloc[0]
    base_row = baseline_summary[baseline_summary["model"] == model_name].iloc[0]
    cluster_params = json.loads(best["cluster_params_json"])

    try:
        built_typ = build_cluster_augmented(train_filt, holdout_typical, best["cluster_name"], cluster_params)
        built_full = build_cluster_augmented(train_filt, holdout_full, best["cluster_name"], cluster_params)
    except Exception:
        built_typ = None
        built_full = None

    if built_typ is None or built_full is None:
        continue

    # --- align feature columns exactly to training matrix ---
    train_cols = list(Xtr_aug.columns)

    Xhold_typ_aug = Xhold_typ_aug.reindex(columns=train_cols, fill_value=0.0)
    Xhold_full_aug = Xhold_full_aug.reindex(columns=train_cols, fill_value=0.0)

    # safety check
    assert list(Xtr_aug.columns) == list(Xhold_typ_aug.columns)
    assert list(Xtr_aug.columns) == list(Xhold_full_aug.columns)

    est = build_regression_model(model_name)
    est.fit(Xtr_aug, ytrain)

    pred_typ = est.predict(Xhold_typ_aug)
    pred_full = est.predict(Xhold_full_aug)

    m_typ = reg_metrics(holdout_typical[TARGET_COL], pred_typ)
    m_full = reg_metrics(holdout_full[TARGET_COL], pred_full)

    best_rows.append({
        "model": model_name,
        "cluster_name": best["cluster_name"],
        "cluster_params_json": best["cluster_params_json"],
        "cv_typical_MAE": float(best["cv_typical_MAE"]),
        "cv_typical_std": float(best["cv_typical_std"]),
        "delta_cv_typical": float(best["delta_cv_typical"]),
        "n_clusters_train": int(best["n_clusters_train"]),
        "use_pca": bool(best["use_pca"]),
        "pca_components": best["pca_components"],
        "fit_mode": best["fit_mode"],
        "n_meta_features": int(best["n_meta_features"]),
        "holdout_typical_MAE": round(m_typ["MAE"], 4),
        "holdout_typical_RMSE": round(m_typ["RMSE"], 4),
        "holdout_typical_MdAE": round(m_typ["MdAE"], 4),
        "holdout_full_MAE": round(m_full["MAE"], 4),
        "holdout_full_RMSE": round(m_full["RMSE"], 4),
        "holdout_full_MdAE": round(m_full["MdAE"], 4),
        "baseline_holdout_typical_MAE": float(base_row["baseline_holdout_typical_MAE"]),
        "baseline_holdout_full_MAE": float(base_row["baseline_holdout_full_MAE"]),
        "delta_holdout_typical": round(float(base_row["baseline_holdout_typical_MAE"]) - m_typ["MAE"], 4),
        "delta_holdout_full": round(float(base_row["baseline_holdout_full_MAE"]) - m_full["MAE"], 4),
    })

cluster_best_by_model = pd.DataFrame(best_rows).sort_values(
    ["cv_typical_MAE", "holdout_typical_MAE"]
).reset_index(drop=True)

display(cluster_best_by_model)

cluster_screen_df.to_csv(ARTIFACT_DIR / "04_cluster_screening_all_baseline_bigsearch_optuna.csv", index=False)
cluster_config_summary_df.to_csv(ARTIFACT_DIR / "04_cluster_config_summary_all_baseline_bigsearch_optuna.csv", index=False)
cluster_best_by_model.to_csv(ARTIFACT_DIR / "04_cluster_best_by_model_all_baseline_bigsearch_optuna.csv", index=False)
baseline_summary.to_csv(ARTIFACT_DIR / "04_cluster_baseline_summary_all_baseline_bigsearch_optuna.csv", index=False)

protocol = {
    "notebook": "ML_meta_cluster",
    "selection_rule": "best CV typical MAE on train_filt using TimeSeriesSplit; no holdout usage for selection",
    "external_holdout": "same 80/20 chronological holdout as baseline notebook",
    "train_definition": "train_full filtered to duration_hours < 960",
    "holdout_typical_definition": "holdout_full filtered to duration_hours < 960",
    "holdout_full_definition": "full external holdout",
    "models": MODEL_ORDER,
    "cluster_methods": list(CLUSTER_EXPERIMENTS.keys()),
    "num_cluster_configs": int(sum(len(v) for v in CLUSTER_EXPERIMENTS.values())),
    "cluster_size_train": "computed only from train labels",
    "confidence_features": [
        "cluster_max_proba",
        "cluster_entropy",
        "cluster_margin_top1_top2",
        "cluster_proba_k",
        "cluster_is_noise",
        "cluster_size_train",
    ],
}
with open(ARTIFACT_DIR / "04_cluster_protocol_all_baseline_bigsearch_optuna.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)

print("Сохранено:")
for name in [
    "04_cluster_screening_all_baseline_bigsearch_optuna.csv",
    "04_cluster_config_summary_all_baseline_bigsearch_optuna.csv",
    "04_cluster_best_by_model_all_baseline_bigsearch_optuna.csv",
    "04_cluster_baseline_summary_all_baseline_bigsearch_optuna.csv",
    "04_cluster_protocol_all_baseline_bigsearch_optuna.json",
]:
    print("-", name)
